# Iris Example
## Understanding the Training Process

This notebook is for those who understand the basics of configuring, training, and evaluating a deep neural network with Zero2Neuro. In this notebook we use the Iris dataset to take a deeper look at the training process for a classification problem.

By the end of this notebook you'll know how to:

- Choose an appropriate loss function for different classification problems
- Split data into training, validation, and test sets
- Monitor training and validation loss
- Use validation performance to monitor and guide model training
- Evaluate a model on held-out test data (final exam!)

# The Problem
The Iris dataset is a old and classic machine learning dataset and offers a new classification problem, multi-class classification. Until now we've only done binary classification but with the Iris dataset there are 3 output classes. We will start with doing a quick examination of the dataset with pandas and then go into the configuration files.

In [ ]:
import os
import sys
# Optional if you don't have the neuro path variable set up in your bashrc (Set to folder/directory above keras3_tools and zero2neuro)
# os.environ["NEURO_REPOSITORY_PATH"] = "/home/myuser/neuro"

neuro_path = os.getenv("NEURO_REPOSITORY_PATH")
assert neuro_path is not None, "Environment variable NEURO_REPOSITORY_PATH must be set to directory above zero2neuro and keras3_tools"

sys.path.append(neuro_path + '/zero2neuro/src/')

from zero2neuro import *
from parser import *

In [ ]:
parser = create_parser()

In [ ]:
import pandas as pd
# ../iris.csv is the default path of the dataset.
df = pd.read_csv("../iris_data.csv")

In [ ]:
# We can see that we have four input columns with float values and then a string for class
df.head()

In [ ]:
# This helps check for null values and confirms that all the rows are uniform in data type and length, we have 150 examples.
df.info()

In [ ]:
# This is very helpful, with pandas you can call specify a column name and use the .unique() function to get the unique values in that column.
# This shows our 3 possible predictions.
df["Class"].unique()

# Cross Validation
Before we go into the data configuration there is another concept that is integral to training and evaluating models: cross validation. For this example we will be using one of the supported type of cross validation in zero2neuro: holistic cross validation. Up until now we have been giving our models the full dataset and training off the entire thing, but this does not simulate how the model would perform on data it hasn't seem before. To fix this problem we use something called cross validation where we split our dataset into three subsets training, validation, and testing.  

Zero2Neuro handles this by taking in a number of folds and then allocating a certain number of folds to training, validation, and testing. With holistic cross validation there is a concept called rotation, which simply will rotate these folds around, see the image below for an example.
  
<img src="../../../images/holistic-n-fold-cross-validation.png" alt="Logo of a cute puppy" width="500">  

These three sets have distinct goals. The training set is used to directly adjust the parameters of the network, the validation set is used to make hyperparameter decisions like adjusting learning rate or layer layout, and the testing set is used as a completely seperate set which is only used for predictions at the very end of training to simulate unseen data.

Here's a metaphor to help illustrate this concept cleaner.
- Training Set: Homework (Learn)
- Validation Set: Practice Exam (Adjust strategy)
- Testing Set: Final Exam (Performance Check)
  
For more information on this subject please look [here](../../../docs/modules/superdataset/data_folds_to_sets.md). Let's move onto the data configuration file.

# Data Config
For this example we've left data configuration file filled out so you can see how these arguments are done.
```
--data_format=tabular

# Desired output value (Class column) is expressed as a string:
#  translate to one of three integers (0,1,2)
--data_columns_categorical_to_int
Class:Iris-setosa,Iris-versicolor,Iris-virginica

# Data Source
--data_files=../iris_data.csv

# Split into 10 folds randomly and perform cross-validation
--data_fold_split=random
--data_set_type=holistic-cross-validation
--data_n_folds=10

# Four features
--data_inputs
sepal length
sepal width
petal length
petal width

# Output column
--data_outputs
Class
```
Like with the breast cancer example we link our class values to integers. To do this cross-validation we must specify the number of folds, how we split the data, and our method of cross calidation. 
  
In this case first we choose random, if the seed that is used doesn't change (by default in zero2neuro its the same) then rotation 0 will always be the same and you can have 10 different rotations. But, if you change this seed that can give you another 10 different rotations.
  
Secondly, we choose holistic cross validation which is the type we showed earlier, where the testing set is also rotated. Then we have data_n_folds=10, by default this means that training will take 8, validation 1, and testing 1.



# Experiment Config
Open up experiment_config.txt  

```
--experiment_name=iris
--data_rotation=0

# There are three output classes so we'll use the loss function
# designed for multiclass classification
--loss=sparse_categorical_crossentropy

# Measures how often the predicted class matches the true class
--metrics=sparse_categorical_accuracy

# TODO: Choose learning rate
# How quickly should network update its parameters?
--learning_rate=TODO

# TODO: Choose the maximum number of training epochs (early stopping can make it less)
--epochs=TODO

# Stop training if the early stopping monitor stops improving
--early_stopping

# The training set is what the model learns directly off of
# The validation set tests the adjustments on a seperate set
# So we use our validation performance to judge whether to stop early or not
--early_stopping_monitor=val_loss

# TODO: Choose how many epochs to wait for improvement before stopping training
--early_stopping_patience=TODO

--results_path=./results
--output_file_base={args.experiment_name}_R{args.data_rotation:02d}
--save_model
--render_model

# Save the training set results
--log_training_set

# Save the validation set results
--log_validation_set

# Save the testing set results
--log_testing_set

# Report the results to a XLSX file
--report
--report_training
--report_training_ins
```

Here we use a new loss and metric function, sparse categorical, which uses integers to represent each true label (our class). There's another multi-class categorical option, the standard categorical, but this uses vectors instead of integers and is mainly for when classes can overlap. We aren't predicting hybrid flowers so we will stick with sparse.  

The second thing that's new is that we are now using val_loss as our early stopping monitor. Before we were using early stopping as a means to save resources and time by stopping when the model wasn't improving in a significant way. Now by using val_loss we can stop the model when it is not generalizing enough to outside data. This happens when a model's training performance is increasing but the validation performance is decreasing, remember that the validation is meant to simulate unseen data, we call this overfitting. We will go into more detail about overfitting and underfitting in the mesonet example, but the takeaway is that we are now stopping short for both performance and for generalization (being able to perform well when applied to data the model hasn't trained on).

# Network Config
```
--network_type=fully_connected

# Four input features:
# sepal length, sepal width, petal length, petal width
--input_shape
4

# TODO: Choose the number of neurons in each hidden layer.
# Experiment with different network sizes and observe the effect
# they have on training and validation performance.
--number_hidden_units
???
???

# TODO: Choose an activation function for the hidden layers.
--hidden_activation=???

# Three output units: one for each iris species.
--output_shape
3

# Softmax converts the three outputs into class probabilities (if there were two options you would use sigmoid)
--output_activation=softmax
```

The only thing that's really new here is softmax, and what this gives is a vector of probabilities, so for three flowers it would be something like: [0.45, 0.05, 0.5]. This would indicate that the model knows its not the 2nd class but is slightly mixed on the first and third, with the third coming on top very slightly. The rest you know how to fill out by now, pick a hidden activation function and experiment with the hidden layers. 

In [ ]:
arg_string = "@network_config.txt @data_config.txt @experiment_config.txt -v --force"
args = parser.parse_args(arg_string.split())
print(args)

In [ ]:
prepare_and_execute_experiment(args)

# Visualizing the Results
Like before if you want an explicit example by example look at the predictions, use the xlsx file.  
  
We have three new visualizations we can do:
- Validation Performance
- Test Performance
- Multi-class confusion matrix

In [ ]:
# We open the pkl file like usual
with open('results/iris_R00_results.pkl', 'rb') as pickle_file:
    data = pickle.load(pickle_file) # Grab the data from pickle

In [ ]:
# Notice we now have validation keys.
print(data.keys())
print(data['history'].keys())

In [ ]:
# As you can see it's a vector, so to grab out predictions we need to write a bit of code to get an integer output of which class.
print(data['predict_validation'][0])

# Learning Curves
Matplotlib has a useful feature where we can set set axes and subplots which allows us to do visualizations side by side. We will use this to visualize our training and validation learning curves and accuracy side by side. 

In [ ]:
# This creates a blank side by side grid for our plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss for both
axes[0].plot(data["history"]["loss"], label="Training")
axes[0].plot(data["history"]["val_loss"], label="Validation")

# Sets up our learning curve plot
axes[0].set_title("Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.5) # Alpha just handles the transparency of the grid

# Accuracy
axes[1].plot(
    data["history"]["sparse_categorical_accuracy"],
    label="Training"
)
axes[1].plot(
    data["history"]["val_sparse_categorical_accuracy"],
    label="Validation"
)

axes[1].set_title("Training and Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.5)

# It's personal preference but tight layout often makes visualizations easier to look at
plt.tight_layout()
plt.show()

The general rule of thumb is that as long as validation's loss is decreasing the model is learning and can continue training. If the training and validation losses diverge that's a sign of overfitting, but early stopping will catch it and return to when the validation performance was best.  

There's a good chance you might notice that while the learning curve is a smooth hill the accuracy is jumping all over the place. The reason for this has to do with how accuracy works vs our loss. Our loss computes how confident the model is in it's prediction, not if it's correct while accuracy only cares if the prediction is correct. For instance take these two predictions where the correct answer is the first class:
[0.51, 0.49, 0.0]  
[0.99, 0.01, 0.0]
Both of these predictions will give an accuracy of 1.0 but the loss for both of them is very different.

# Confusion Matrix
Now for the confusion matrix. Since we have a test set we only really need to worry about that one as that gives us the best idea of how our model is doing on completely unseen data. You might also notice we don't have a testing curve, that is because the testing set is only used at the very end (basically for 1 epoch) so there's no curve to plot.

In [ ]:
# Our pickle file includes our label translations, just to double check.
print(data['dataset'].keys())
print(data['dataset']['categorical_translation'])
# Grab our class names (this can be automated with a bit of code also)
CLASS_NAMES = ["Setosa", "Versicolor", "Virginica"]

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

actual = data["outs_testing"]

# Our model predicts a probability, to plot we need a flat integer so we set a threshold so that anything over 0.5
# is a 1 and anything below is a 0.
predicted = np.argmax(data["predict_testing"], axis=1)

cm = confusion_matrix(actual, predicted)

# Here we set our xtick and ytick labels. Make sure you match these up with the categorical translations in data config.
# A fun thing to try out is changing the cmpap (this changes the color, real iris flowers a purple and yellow but unfortunately it isn't an option)
sns.heatmap(cm, annot=True, fmt="d", cmap="BuPu", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

That's all for the iris example. The confusion matrix is very small as we split a dataset of just 150 examples into 10 different pieces, but our next example, mesonet, will not have that issue. 